# 01 - The DECAL detector and its data  (student scaffold)

Welcome to CALOMAPS. This first notebook is a guided tour of **what we simulate
and what comes out**: open one simulation file, understand the detector geometry
that *quantizes* every shower into 100 µm pixels, watch a single shower land on
that geometry, and make a few first-look plots.

DECAL = **D**igital **E**lectromagnetic **CAL**orimeter: instead of measuring
analog charge, the silicon is finely pixelated (100 µm) and each pixel is
essentially a *yes/no* hit. Counting hits ~ counting particles -- until the shower
gets so dense that pixels saturate. Exploring that trade-off is the whole
project, and it all starts from the geometry and data shown here.

**This is the scaffold notebook.** Cells marked `# TODO (your task)` are yours to
fill in -- they `raise NotImplementedError` until you do. The geometry parse, the
file-open boilerplate, and the plotting scaffolding are given; the science (the
hit selection and the key plots) is up to you. Each task states a **hint** and an
**expected** result so you can sanity-check yourself; the worked solution lives in
`notebooks/01` (photon) / `01b` (pion) on the `main` branch if you get stuck.

**Prerequisites**: `setup_calomaps.sh` sourced, and some simulation data present
(see [`docs/handbook.md`](../docs/handbook.md) §8 to generate a smoke-test sim).

**Kernel**: `Python (Key4hep)` -- no GPU needed.

## 0. Choose your particle (junior = photons, senior = pions)

Two students run **this same notebook** with a **different particle** and compare
results at the end. Set `PARTICLE` once here; everything downstream keys off it.

- **Junior** → `PARTICLE = "gamma"` (photons: clean, compact, *electromagnetic* showers)
- **Senior** → `PARTICLE = "pi+"` (pions: wide, long, fluctuating, *hadronic* — often a
  MIP track punching through this *EM* calorimeter)

Keep an eye on how the two look different at every plot below — that contrast is the point.

In [ ]:
# ---- THE ONE KNOB THAT SWITCHES STUDENTS -------------------------------------
PARTICLE = "gamma"   # junior: "gamma";  senior: "pi+"

# Dataset + file-glob pattern both derive from PARTICLE, so the rest of the
# notebook is particle-agnostic. (The pion production lives in a parallel dataset.)
DATASET = "data_spectrum_100um_400GeV" if PARTICLE == "gamma" else "data_spectrum_100um_400GeV_piplus"
FILE_GLOB = "sim_photons_part*.root" if PARTICLE == "gamma" else "sim_piplus_part*.root"

print(f"PARTICLE  = {PARTICLE}")
print(f"DATASET   = {DATASET}")
print(f"FILE_GLOB = {FILE_GLOB}")

## 1. Find a data file

In [ ]:
import os
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import glob

# CALOMAPS_DATA_BASE points to where simulation output lives (set by setup_calomaps.sh).
data_base = os.environ.get("CALOMAPS_DATA_BASE", os.path.expanduser("~/CALOMAPS-data"))
dataset_dir = os.path.join(data_base, DATASET)   # DATASET from the config cell above

# Pick the first file (FILE_GLOB also derives from PARTICLE)
files = sorted(glob.glob(os.path.join(dataset_dir, FILE_GLOB)))
print(f"found {len(files)} files in {dataset_dir}")
print(f"opening first one: {files[0] if files else '(none -- generate data first)'}")
filepath = files[0]


## 2. What's inside the file?

The file is a ROOT file. Inside is a `TTree` named `events` (one entry per simulated event), with a flat list of branches that pack everything: truth-particle info, hit positions, hit energies, layer IDs, cell IDs, etc.

`uproot` lets us read these without depending on the C++ ROOT install.


In [ ]:
with uproot.open(filepath) as f:
    print("top-level keys in the file:")
    for k in f.keys()[:8]:
        print(f"  {k}")
    print("...")
    tree = f["events"]
    print(f"\nevents.num_entries = {tree.num_entries}")
    print(f"number of branches  = {len(tree.keys())}")
    print(f"\nfirst 10 branches:")
    for k in tree.keys()[:10]:
        print(f"  {k}")


## 3. The detector geometry — a *quantized* calorimeter

The ECal is a **12-sided silicon barrel** (a dodecagonal tube) whose axis runs
along **z**. Radially it is a sandwich of **30 silicon sensor layers** interleaved
with tungsten absorber, between r = 1264 mm (inner face) and r = 1403 mm. Two
kinds of quantization define a DECAL:

- **Transverse**: each layer is tiled into **100 µm pixels** — the spatial quantum.
  A hit is a *pixel*, not a continuous position.
- **Longitudinal**: the shower is sampled at **30 discrete depths** (the layers).

We parse the layer radii straight out of the geometry XML below, and reuse them
to overlay the detector on the data.

In [ ]:
import os, glob, xml.etree.ElementTree as ET
import uproot, awkward as ak, numpy as np, matplotlib.pyplot as plt

CALOMAPS_HOME = os.environ.get("CALOMAPS_HOME", os.path.expanduser("~/CALOMAPS"))
DATA_BASE = os.environ.get("CALOMAPS_DATA_BASE", os.path.expanduser("~/CALOMAPS-data"))

def parse_sid_value(v, constants):
    if not v: return 0.0
    if v in constants: return parse_sid_value(constants[v], constants)
    s = v.replace('*', ' * ').replace('cm', '10').replace('mm', '1')
    try: return eval(s, {"__builtins__": None}, {})
    except: return float(v)

geom_dir = os.path.join(CALOMAPS_HOME, "geometry")
main_xml = ET.parse(os.path.join(geom_dir, "SiD_TestBeam.xml")).getroot()
consts = {c.get("name"): c.get("value") for c in main_xml.findall(".//constant")}
custom_xml = ET.parse(os.path.join(geom_dir, "my_custom_ecal.xml")).getroot()
det = custom_xml.find(".//detector[@name='ECalBarrel']")

rmin = parse_sid_value(det.find("dimensions").get("rmin"), consts)
print(f"Detector face (rmin): {rmin:.2f} mm")

sensor_y_planes = []
curr_y = rmin
for layer in det.findall("layer"):
    rep = int(layer.get("repeat", 1))
    slices = layer.findall("slice")
    thick = sum(parse_sid_value(s.get("thickness"), consts) for s in slices)
    si_off = 0
    off = 0
    for s in slices:
        t = parse_sid_value(s.get("thickness"), consts)
        if s.get("material") == "Silicon":
            si_off = off + t / 2.0
        off += t
    for _ in range(rep):
        sensor_y_planes.append(curr_y + si_off)
        curr_y += thick

print(f"Total silicon layers: {len(sensor_y_planes)}")
print(f"First 5 layers at y = {[f'{y:.2f}' for y in sensor_y_planes[:5]]} mm")
print(f"Last 5 layers at y  = {[f'{y:.2f}' for y in sensor_y_planes[-5:]]} mm")


## 4. Look at a single event

Each event is one particle (your `PARTICLE` — `gamma` or `pi+`) fired from the origin in **+y**, interacting in the
silicon. The hit collection is *jagged* (a different number of hits per event);
`awkward` handles that. Let's pull the first event's hits and truth energy.

In [ ]:
with uproot.open(filepath) as f:
    tree = f["events"]
    # Read the first event only
    x = tree["ECalBarrelHits/ECalBarrelHits.position.x"].array(entry_stop=1)[0]
    y = tree["ECalBarrelHits/ECalBarrelHits.position.y"].array(entry_stop=1)[0]
    z = tree["ECalBarrelHits/ECalBarrelHits.position.z"].array(entry_stop=1)[0]
    e = tree["ECalBarrelHits/ECalBarrelHits.energy"].array(entry_stop=1)[0]
    # Truth particle: the incident particle's true energy
    p_mc = tree["MCParticles/MCParticles.momentum.y"].array(entry_stop=1)[0]
    print(f"Event 0 ({PARTICLE}):")
    print(f"  truth {PARTICLE} momentum.y (= |p| for a +Y beam): {float(p_mc[0]):.2f} GeV")
    print(f"  number of pixel hits: {len(x)}")
    print(f"  hit x range:  [{ak.min(x):.1f}, {ak.max(x):.1f}] mm")
    print(f"  hit y range:  [{ak.min(y):.1f}, {ak.max(y):.1f}] mm   (expect ~1264-1403 for the +Y barrel face)")
    print(f"  hit z range:  [{ak.min(z):.1f}, {ak.max(z):.1f}] mm")
    print(f"  hit energy range: [{ak.min(e)*1e6:.2f}, {ak.max(e)*1e6:.0f}] keV")
    print(f"  total visible E: {float(ak.sum(e)):.4f} GeV   (visible/true = the sampling fraction; measured in section 5)")


## 5. Visualize the event on the detector

The natural way to *see the whole detector* is the **transverse plane (x–y)** —
perpendicular to the beam axis — where the full 12-sided ring is visible. The
photon travels **outward from the origin** and enters the **inner** face
(r = 1264 mm).

Two views:
1. **Transverse cross-section (x–y)** — the whole barrel ring. Most hits land on
   the +y entry face, but a single shower isn't fully contained: soft secondaries
   leak across the inner air cavity and strike the other faces too — which is why
   hits appear all the way around the ring.
2. **Zoom on the +y entry face** — the same plane, with the 30 silicon layers
   drawn. Increasing radius is increasing depth, so this is where you watch the
   shower develop layer by layer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import RegularPolygon

x_np = ak.to_numpy(x); y_np = ak.to_numpy(y); z_np = ak.to_numpy(z)
e_keV = ak.to_numpy(e) * 1e6   # GeV -> keV for the colour scale

# Barrel geometry (from geometry/SiD_TestBeam.xml): 12-sided, r = 1264..1403 mm,
# axis along z. The ring therefore lives in the x-y plane.
NSIDES, RMIN, RMAX = 12, 1264.0, 1403.0

def dodecagon(apothem):
    # RegularPolygon wants the circumradius (centre->vertex) = apothem / cos(pi/n);
    # rotate by pi/n so a flat FACE (not a vertex) points +y, matching the entry face.
    return RegularPolygon((0, 0), numVertices=NSIDES,
                          radius=apothem / np.cos(np.pi / NSIDES),
                          orientation=np.pi / NSIDES,
                          fill=False, edgecolor="0.5", lw=1.2)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Panel 1 — full transverse ring (x-y), all hits + the 12-sided silicon shell
ax = axes[0]
ax.add_patch(dodecagon(RMIN)); ax.add_patch(dodecagon(RMAX))
sc = ax.scatter(x_np, y_np, c=e_keV, cmap="viridis", s=2, alpha=0.5, norm=LogNorm())
# the particle starts at the ORIGIN and travels +y outward, entering the INNER face
ax.annotate("", xy=(0, RMIN), xytext=(0, 560),
            arrowprops=dict(arrowstyle="-|>", color="red", lw=2.5))
ax.text(90, 300, f"{PARTICLE}:\norigin -> +y", color="red", fontsize=9, va="center")
ax.set_aspect("equal"); ax.set_xlim(-1700, 1700); ax.set_ylim(-1700, 1700)
ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
ax.set_title("Transverse cross-section (x-y):\nthe 12-sided silicon barrel")
fig.colorbar(sc, ax=ax, label="hit energy [keV]")

# Panel 2 — zoom on the +y entry face, with the silicon layers as lines
ax = axes[1]
face = (y_np > 1240) & (np.abs(x_np) < 70)
layer_y = np.unique(np.round(y_np[face], 1))
for ly in layer_y:
    ax.axhline(ly, color="0.85", lw=0.6, zorder=0)
sc2 = ax.scatter(x_np[face], y_np[face], c=e_keV[face], cmap="viridis", s=5,
                 alpha=0.7, norm=LogNorm())
ax.set_xlim(-70, 70); ax.set_ylim(1255, 1410)
ax.set_xlabel("x [mm]"); ax.set_ylabel("y (depth into calorimeter) [mm]")
ax.set_title(f"Zoom on the +y entry face:\n{len(layer_y)} silicon layers")
fig.colorbar(sc2, ax=ax, label="hit energy [keV]")

plt.tight_layout()
plt.show()
print(f"Left: the whole shower in the barrel cross-section -- most hits on the +y face, "
      f"the rest is leakage across the cavity.")
print(f"Right: zoomed to the entry face; the {len(layer_y)} horizontal lines are the "
      f"silicon sensor layers the shower crosses.")

## 6. How the shower develops through the detector

Now follow this one shower *into* the calorimeter. We keep the +y entry segment,
assign every hit to its silicon layer (by radius), and look at three views:

1. **One layer at 100 µm resolution** — zoom into the densest layer so the
   individual pixels are visible. This *is* the quantization: each lit square is
   a 100 µm × 100 µm yes/no pixel.
2. **Layer-by-layer evolution** — the transverse hit pattern in all 30 layers,
   showing the shower start narrow, fan out at shower-max, then fade.
3. **Longitudinal profile** — total energy per layer vs depth for this single
   event (compare to the energy-averaged version in `02_data_extraction`).

In [ ]:
import numpy as np
e_np = ak.to_numpy(e)                       # this event's hit energies (GeV)
RADII = np.array(sensor_y_planes)           # 30 silicon layer radii from the XML

# TODO (your task): isolate the hits from the +y entry wedge, then assign each
#   surviving hit to its silicon layer. You must end this cell with these arrays
#   defined (the plots below all use them):
#     xs, zs, es, rs : the x, z, energy, radius of every hit inside the +y wedge
#     lyr            : integer layer index (0..29) for each of those hits
#
# Hint: a hit's radius is r = hypot(x, y); its azimuth around the +y axis is
#   angle = degrees(arctan2(x, y)). Keep |angle| < 15 deg (a 30-deg wedge centred on
#   +y) AND 1260 < r < 1410 mm (between the inner and outer silicon faces). Then map
#   each hit to the NEAREST layer radius: lyr = argmin over RADII of |RADII - r|
#   (broadcast: np.argmin(np.abs(RADII[None,:] - rs[:,None]), axis=1)).
# Expected: roughly 4-5 x 10^4 hits survive for a ~400 GeV photon, spread over all
#   30 layers. (Pions: fewer hits, possibly missing some layers -- that's the point.)
# Solution: see notebooks/01 (photon) / 01b (pion) on the 'main' branch.
raise NotImplementedError("fill this in: build xs, zs, es, rs, lyr for the +y wedge")

# (sanity check your filled-in cell — once xs/lyr exist this should print)
print(f"{len(xs)} hits in the +y wedge, spread over {len(np.unique(lyr))} layers")


In [ ]:
# Plot (d): the densest layer, rendered at native 100 um pixel resolution, so each
# lit square IS one 100 um x 100 um pixel -- the spatial quantum of the DECAL.
PITCH = 0.1   # mm per pixel (100 um)

# TODO (your task): build a 2-D image `img` of per-pixel energy (keV) for the
#   single densest layer, centred on the shower core. You must define, for the
#   plotting block below:
#     dense        : index of the layer with the most hits
#     img          : (nz, nx) array of summed hit energy per 100 um pixel (NaN = empty)
#     xed, zed     : the pixel-EDGE coordinates in mm (length nx+1 / nz+1) for pcolormesh
#
# Hint: dense = np.bincount(lyr, minlength=len(RADII)).argmax(). Keep that layer's hits
#   (m = lyr == dense). Convert positions to integer pixel indices: xi = round(xs[m]/PITCH),
#   zi = round(zs[m]/PITCH); energies ei = es[m]*1e6 (keV). Pick a window half = 15 pixels
#   (+-1.5 mm) around the median (xi, zi), accumulate energy into img[jz, jx], and build
#   edges as (arange(lo, hi+2) - 0.5) * PITCH so each cell spans exactly one pixel.
# Expected: a compact bright core a few pixels across for photons (one tight EM blob);
#   for pions, expect it sparser / off-centre / possibly multi-core.
# Solution: see notebooks/01 (photon) / 01b (pion) on the 'main' branch.
raise NotImplementedError("fill this in: build dense, img, xed, zed")

# --- plotting (given: produces plot (d) once img/xed/zed exist) ---
fig, ax = plt.subplots(figsize=(6.5, 6))
pcm = ax.pcolormesh(xed, zed, img, cmap="magma")
ax.set_xticks(xed, minor=True); ax.set_yticks(zed, minor=True)
ax.grid(which="minor", color="0.7", lw=0.5)
ax.set_aspect("equal")
ax.set_xlabel("x [mm]"); ax.set_ylabel("z [mm]")
ax.set_title(f"Densest layer (#{dense}) at 100 µm resolution\n(each coloured square = one 100 µm pixel)")
fig.colorbar(pcm, ax=ax, label="hit energy [keV]")
plt.tight_layout(); plt.show()


In [ ]:
# Plot (e): the transverse hit pattern in each of the 30 layers, so you watch the
# shower start narrow, fan out at shower-max, then fade with depth.
nlayers = len(RADII)
fig, axes = plt.subplots(5, 6, figsize=(15, 12), sharex=True, sharey=True)
vmax = (es * 1e6).max() if len(es) else 1.0   # common colour scale across layers

# TODO (your task): for each layer i in 0..29, draw that layer's hits in the x-z
#   plane on subplot axes.flat[i], coloured by hit energy (keV) on the shared vmax.
# Hint: loop `for i, ax in enumerate(axes.flat)`; for i >= nlayers do ax.axis('off').
#   Select that layer with m = lyr == i; if m.any(): ax.scatter(xs[m], zs[m],
#   c=es[m]*1e6, cmap='magma', s=4, vmin=0, vmax=vmax). Set ax.set_xlim(-30, 30),
#   ax.set_ylim(-30, 30) and a small title ax.set_title(f'L{i}', fontsize=8).
# Expected: photons -> a tight spot that brightens around layers ~5-12 (shower-max)
#   then dims; pions -> patchier, wider, sometimes just a single MIP dot per layer.
# Solution: see notebooks/01 (photon) / 01b (pion) on the 'main' branch.
raise NotImplementedError("fill this in: draw the per-layer x-z hit scatter for all 30 layers")

fig.suptitle("Transverse shower evolution through the 30 silicon layers (x vs z per layer)", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# Plot (f): the longitudinal profile -- total energy deposited in each layer vs depth.
nlayers = len(RADII)

# TODO (your task): compute `le`, the total hit energy (GeV) in each of the 30 layers,
#   ordered by layer index (le[i] = sum of es over hits with lyr == i).
# Hint: np.bincount(lyr, weights=es, minlength=nlayers)[:nlayers] does this in one line.
# Expected: a rise-then-fall curve peaking near shower-max for photons (a few layers in);
#   pions deposit deeper and flatter, with bigger layer-to-layer fluctuations.
# Solution: see notebooks/01 (photon) / 01b (pion) on the 'main' branch.
raise NotImplementedError("fill this in: build le, the per-layer energy sum")

# --- plotting (given: produces plot (f) once le exists) ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(RADII, le, width=2.5, color="royalblue", edgecolor="black", alpha=0.8)
ax.plot(RADII, le, color="darkorange", marker="o", lw=2)
if nlayers > 20:
    ax.axvline(RADII[20], color="red", ls="--", alpha=0.7, label="thin->thick W transition")
    ax.legend()
ax.set_xlabel("layer radius (depth) [mm]"); ax.set_ylabel("energy in layer [GeV]")
ax.set_title("Longitudinal shower profile of this single event")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 7. Reverse-engineer the geometry from the data

A nice sanity check: we should be able to *recover* the pixel pitch and the layer
count straight from the hit positions and confirm they match the XML — proving
the simulation quantizes the way we expect.

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_BASE, DATASET, FILE_GLOB)))
filepath = files[0]
print(f"opening {filepath}")

with uproot.open(filepath) as f:
    tree = f["events"]
    x_j = tree["ECalBarrelHits/ECalBarrelHits.position.x"].array()
    y_j = tree["ECalBarrelHits/ECalBarrelHits.position.y"].array()
    z_j = tree["ECalBarrelHits/ECalBarrelHits.position.z"].array()

x_all = ak.flatten(x_j).to_numpy()
y_all = ak.flatten(y_j).to_numpy()
z_all = ak.flatten(z_j).to_numpy()

# Filter to the +Y face only (where the particles enter)
mask = (np.abs(x_all) <= 50) & (np.abs(z_all) <= 50) & (y_all >= 1260) & (y_all <= 1420)
x, y, z = x_all[mask], y_all[mask], z_all[mask]
print(f"Total hits in file:        {len(x_all)}")
print(f"Hits in +Y target volume:  {len(x)}")


In [ ]:
unique_x = np.unique(np.round(x, 4))
unique_y = np.unique(np.round(y, 4))
unique_z = np.unique(np.round(z, 4))

dx = np.diff(unique_x)
dz = np.diff(unique_z)
pitch_x = np.min(dx[dx > 0]) if len(dx) > 0 else 0
pitch_z = np.min(dz[dz > 0]) if len(dz) > 0 else 0

print(f"Pixel X pitch (from data):  {pitch_x:.5f} mm    (expect 0.1 mm = 100 um)")
print(f"Pixel Z pitch (from data):  {pitch_z:.5f} mm")
print(f"Sensor layers (from data):  {len(unique_y)}      (expect 30 from XML)")
print(f"XML-derived layer y values match data:",
      np.allclose(np.array(sensor_y_planes), unique_y[:len(sensor_y_planes)], atol=0.01))


## 8. First-look aggregates

Finally, look across all events in this one file: how the digital (hit count) and
analog (visible energy) readouts grow with true energy, and what the **sampling
fraction** (visible/true) actually is for this design — *measured* from the data,
not assumed.

In [ ]:
with uproot.open(filepath) as f:
    tree = f["events"]
    x_all = tree["ECalBarrelHits/ECalBarrelHits.position.x"].array()
    e_all = tree["ECalBarrelHits/ECalBarrelHits.energy"].array()
    p_all = tree["MCParticles/MCParticles.momentum.y"].array()

hits_per_event = ak.num(x_all)
total_e_per_event = ak.sum(e_all, axis=1)
true_e_per_event = ak.flatten(p_all)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(true_e_per_event, bins=20)
axes[0].set_xlabel(f"true {PARTICLE} energy [GeV]")
axes[0].set_ylabel("events")
axes[0].set_title("Truth energy spectrum")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(true_e_per_event, hits_per_event, s=20)
axes[1].set_xlabel(f"true {PARTICLE} energy [GeV]")
axes[1].set_ylabel("number of pixel hits")
axes[1].set_title("Hits vs E_true (digital readout)")
axes[1].grid(True, alpha=0.3)

axes[2].scatter(true_e_per_event, total_e_per_event, s=20)
# The sampling fraction is design-dependent (Si/W thickness ratio, layer count, ...),
# so we MEASURE it from the data instead of assuming a fixed value. Fit through the
# origin: E_visible = f * E_true. Change the geometry and this number changes.
_t = np.asarray(true_e_per_event, dtype=float)
_v = np.asarray(total_e_per_event, dtype=float)
samp_frac = float(np.sum(_t * _v) / np.sum(_t * _t))
_xline = np.array([0.0, _t.max()])
axes[2].plot(_xline, samp_frac * _xline, "k--", alpha=0.7,
             label=f"fit: {samp_frac*100:.2f}% sampling fraction")
axes[2].set_xlabel(f"true {PARTICLE} energy [GeV]")
axes[2].set_ylabel("visible energy [GeV]")
axes[2].set_title("E_visible vs E_true (analog readout)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## What you just saw — the physics in one paragraph

The DECAL is a **quantized sampling calorimeter**: a 12-sided silicon barrel,
30 layers deep, every layer tiled into 100 µm yes/no pixels. A single particle (your `PARTICLE`)
fired in +y deposits energy in it — you saw the same event three ways: spread
around the *whole ring* (entry face + cross-cavity leakage), developing
*layer by layer* (for an EM shower: narrow → shower-max → fade; a pion may instead
punch through as a sparse MIP track — compare with your partner!), and *pixel by pixel* at 100 µm.
Two physics threads run through the rest of the project:

- **Sampling**: only a small fraction of the particle's energy lands in the silicon (the rest
  is absorbed in the tungsten) — the *sampling fraction*, set by the Si/W
  geometry, and the source of the **stochastic** resolution term. For photons it's
  ~1.4%; for pions it's lower and highly variable — you measure your own in section 8.
- **Quantization**: counting lit pixels (digital) tracks energy at low E, but at
  high E the shower core puts many particles through one pixel, which can only
  fire once — **saturation**. Whether the digital readout's resolution beats or
  loses to analog, and where the crossover sits, is exactly what
  [`02_data_extraction.ipynb`](02_data_extraction.ipynb) and
  [`03_ml_training_and_eval.ipynb`](03_ml_training_and_eval.ipynb) quantify.

## What's next

You've seen the detector, one shower on it, and the first-look readouts. Next:

- [`02_data_extraction.ipynb`](02_data_extraction.ipynb) reduces every event to
  five readout numbers (isolating the entry segment, with the analog / MIP /
  hits / cluster definitions).
- [`03_ml_training_and_eval.ipynb`](03_ml_training_and_eval.ipynb) trains the
  Deep Quantile Ensemble surrogate and runs the Neyman-inversion reconstruction.

For physics context see [`docs/DECAL_pipeline.md`](../docs/DECAL_pipeline.md) and
[`docs/handbook.md`](../docs/handbook.md).